# SEED-VII EEGNet × LoRA-LLM — 本地可编程 Pipeline

本 Notebook 在**本地环境**中运行完整 Pipeline。
与 ModelScope/Kaggle 版本的关键区别：**所有路径均可通过顶部变量配置**，
支持现有本地数据 / 从 ModelScope 拉取两种模式。

## 使用方式

1. 修改下方 `## 用户配置区 ##` 的路径变量
2. 依次执行各 Cell

> **设计原则**：Notebook 通过 `git clone` 关联 GitHub 仓库而非将代码嵌入数据集。
> 所有中间产物（NPZ、checkpoints）均落盘，已存在则自动跳过。

## 0. 用户配置区（请在此修改路径）

In [ ]:
from pathlib import Path
import os

# ═══════════════════════════════════════════════════════════
#  用户可编程路径 — 按需修改以下变量
# ═══════════════════════════════════════════════════════════

# ── 仓库路径 ──
# 如果已克隆过仓库，直接指向本地路径；否则设为 None，脚本会自动 clone
LOCAL_REPO_PATH = None   # 例如: Path('/home/user/EEG_OPUS')  或 None（自动 clone）
GIT_CLONE_DIR   = Path('./EEG_OPUS')   # 自动 clone 时的目标目录

# ── 工作根目录 ──
# 所有数据、模型、输出均存放在此目录下
WORK_ROOT = Path('./workspace')            # 可改为 /mnt/data/seedvii 等

# ── 数据集配置 ──
DATASET_ID            = 'DEREKVERSE/SEED-VII'
LOCAL_DATASET_DIR     = WORK_ROOT / 'seedvii_ms_dataset'      # 原始数据集落盘位置
USE_EXISTING_DATASET  = False   # True: 跳过下载，直接使用 LOCAL_DATASET_DIR 中已有数据

# ── NPZ 预处理输出 ──
NPZ_DIR = WORK_ROOT / 'seedvii_npz'

# ── 训练输出 ──
RUN_DIR = WORK_ROOT / 'runs' / 'run_valence3'

# ── LLM 模型路径 ──
# 若本地已有模型目录，填入绝对路径；否则设为 None 自动下载
LLM_MODEL_DIR         = None   # 例如: Path('/home/user/models/Qwen2.5-0.5B-Instruct')
LLM_MODEL_ID          = 'Qwen/Qwen2.5-0.5B-Instruct'
LLM_CACHE_DIR         = WORK_ROOT / 'models'   # 自动下载时的缓存目录

# ── 预处理参数 ──
PREPROCESS_SUBJECTS   = '1-20'
PREPROC_WINDOW_SEC    = 4.0
PREPROC_STRIDE_SEC    = 4.0
PREPROC_CENTER_RATIO  = 0.60
PREPROC_MAX_WINDOWS   = 12
PREPROC_SHARD_SIZE    = 512

# ── 训练参数 ──
TRAIN_BATCH_SIZE      = 96      # 按 GPU 显存调整
TRAIN_EPOCHS          = 50
TRAIN_STEPS_PER_EPOCH = 300
TRAIN_NUM_WORKERS     = 4       # DataLoader workers, CPU 设 0
TRAIN_DEVICE          = 'auto'  # 'auto' / 'cuda' / 'cpu'
TRAIN_RESUME          = True    # 自动从 last.pt 恢复

# ── ModelScope Token（私有数据集时设置）──
MODELSCOPE_TOKEN = os.environ.get('MODELSCOPE_TOKEN', None)

# ═══════════════════════════════════════════════════════════
print('[CONFIG] 配置加载完成。确认上述路径无误后继续。')
for d in [WORK_ROOT, LOCAL_DATASET_DIR, NPZ_DIR, RUN_DIR, LLM_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. 环境准备：克隆仓库 + 安装依赖

In [ ]:
import sys
from pathlib import Path

print('Python:', sys.version)
CWD = Path.cwd()
print('CWD:', CWD)

# ── 定位 / 克隆仓库 ──
if LOCAL_REPO_PATH is not None:
    REPO = Path(LOCAL_REPO_PATH)
    assert REPO.exists(), f'指定的本地仓库不存在: {REPO}'
else:
    REPO = Path(GIT_CLONE_DIR).resolve()
    if not (REPO / 'seedvii_modal_contrastive_lora' / 'pyproject.toml').exists():
        print(f'[CLONE] Cloning into {REPO} ...')
        !git clone https://github.com/PRIMOCOSMOS/EEG_OPUS.git {REPO}
    else:
        print(f'[SKIP] Repo exists at {REPO}')

PROJECT = REPO / 'seedvii_modal_contrastive_lora'
assert (PROJECT / 'pyproject.toml').exists(), f'Repo structure error: {PROJECT}'
print('PROJECT:', PROJECT)

In [ ]:
# ── 安装依赖 ──
%pip install -q -r {PROJECT / 'requirements.txt'}
%pip install -q modelscope
%pip install -q -e {PROJECT}

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print('[OK] Dependencies installed')

## 2. 数据集获取

两种模式：
- `USE_EXISTING_DATASET=False`：通过 ModelScope API 下载
- `USE_EXISTING_DATASET=True`：直接使用本地已有数据

In [ ]:
import subprocess
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths
from seedvii_contrastive.data.discovery import SUBJECT_FILE_NAMES

def _count_mat_files(dir_path):
    p = Path(dir_path)
    if not p.exists():
        return 0
    return len([f for f in p.glob('*.mat') if f.name in SUBJECT_FILE_NAMES])

if not USE_EXISTING_DATASET:
    # 检查是否已完成下载
    eeg_root_test, text_csv_test = find_downloaded_paths(LOCAL_DATASET_DIR)
    if eeg_root_test is not None and _count_mat_files(eeg_root_test) >= 20:
        print(f'[SKIP] Dataset exists with {_count_mat_files(eeg_root_test)} .mat files')
    else:
        print(f'[DOWNLOAD] Fetching {DATASET_ID} via ModelScope ...')
        token_flag = f'--token {MODELSCOPE_TOKEN}' if MODELSCOPE_TOKEN else ''
        cmd = (
            f'python -m seedvii_contrastive.scripts.download_modelscope_seedvii '
            f'--dataset-id {DATASET_ID} '
            f'--local-dir {LOCAL_DATASET_DIR} '
            f'--max-workers 4 {token_flag}'
        )
        !{cmd}
else:
    print(f'[USE EXISTING] Using local dataset at: {LOCAL_DATASET_DIR}')
    assert LOCAL_DATASET_DIR.exists(), f'指定的本地数据集目录不存在: {LOCAL_DATASET_DIR}'

In [ ]:
# ── 自动发现 EEG_ROOT 和 TEXT_CSV ──
EEG_ROOT, TEXT_CSV = find_downloaded_paths(LOCAL_DATASET_DIR)
print(f'EEG_ROOT = {EEG_ROOT}')
print(f'TEXT_CSV = {TEXT_CSV}')

if EEG_ROOT is None:
    raise FileNotFoundError(
        f'未找到 1-20.mat 所在目录。请确认 LOCAL_DATASET_DIR 指向正确路径: {LOCAL_DATASET_DIR}'
    )
if TEXT_CSV is None:
    raise FileNotFoundError(
        f'未找到 text_protocol*.csv。LLM Tower 必须使用 L2 文本协议。'
        f'请在 {LOCAL_DATASET_DIR} 中放置 text_protocol.csv'
    )

## 3. LLM 模型加载

优先级：
1. `LLM_MODEL_DIR` (用户指定本地路径)
2. ModelScope 缓存 (`LLM_CACHE_DIR`)
3. 自动从 ModelScope 下载

In [ ]:
def _looks_like_transformers_model(path: Path) -> bool:
    return path.exists() and (path / 'config.json').exists()

resolved_model_dir = None

# ── 优先级 1: 用户指定 ──
if LLM_MODEL_DIR is not None:
    p = Path(LLM_MODEL_DIR)
    if _looks_like_transformers_model(p):
        resolved_model_dir = p
    elif p.exists():
        # 搜索子目录
        hits = list(p.rglob('config.json'))
        if hits:
            resolved_model_dir = hits[0].parent
            print(f'[LLM] Found model under user-specified dir: {resolved_model_dir}')

# ── 优先级 2: 检查缓存 ──
if resolved_model_dir is None:
    for candidate in sorted(LLM_CACHE_DIR.rglob('config.json'), key=lambda x: len(str(x))):
        if 'Qwen' in str(candidate) or 'qwen' in str(candidate):
            resolved_model_dir = candidate.parent
            print(f'[LLM] Found cached model: {resolved_model_dir}')
            break

# ── 优先级 3: 从 ModelScope 下载 ──
if resolved_model_dir is None:
    print(f'[LLM] Downloading {LLM_MODEL_ID} from ModelScope ...')
    from modelscope import snapshot_download
    download_path = snapshot_download(LLM_MODEL_ID, cache_dir=str(LLM_CACHE_DIR))
    resolved_model_dir = Path(download_path)

print(f'[LLM] Using model: {resolved_model_dir}')
assert _looks_like_transformers_model(resolved_model_dir), \
    f'LLM 模型目录无效: {resolved_model_dir}'

MODEL_DIR = resolved_model_dir

## 4. NPZ 预处理

若 `index.csv` 已存在则跳过。如需强制重处理，删除 `NPZ_DIR` 目录后重新运行。

In [ ]:
if not (NPZ_DIR / 'index.csv').exists():
    print('[PREPROC] Starting NPZ preprocessing ...')
    !python -m seedvii_contrastive.scripts.preprocess_npz \
        --input-root {EEG_ROOT} \
        --output-dir {NPZ_DIR} \
        --subjects {PREPROCESS_SUBJECTS} \
        --window-sec {PREPROC_WINDOW_SEC} \
        --stride-sec {PREPROC_STRIDE_SEC} \
        --center-ratio {PREPROC_CENTER_RATIO} \
        --max-windows-per-clip {PREPROC_MAX_WINDOWS} \
        --shard-size {PREPROC_SHARD_SIZE}
else:
    print(f'[SKIP] NPZ index exists: {NPZ_DIR / "index.csv"}')
    print(f'       如需重处理请删除 {NPZ_DIR}')

## 5. 写入运行配置

从 `modelscope_default.yaml` 模板出发，将所有用户可配置参数注入。

In [ ]:
import yaml

base_cfg_path = PROJECT / 'configs' / 'modelscope_default.yaml'
cfg = yaml.safe_load(open(base_cfg_path, 'r', encoding='utf-8'))

# ── 数据路径 ──
cfg['data']['modelscope_dataset_id'] = DATASET_ID
cfg['data']['local_dataset_dir']     = str(LOCAL_DATASET_DIR)
cfg['data']['eeg_root']              = str(EEG_ROOT)
cfg['data']['text_csv_path']         = str(TEXT_CSV)
cfg['data']['npz_dir']               = str(NPZ_DIR)

# ── 运行时 ──
cfg['runtime']['output_dir'] = str(RUN_DIR)
cfg['runtime']['device']     = TRAIN_DEVICE

# ── LLM 模型 ──
cfg['model']['llm']['model_name_or_path'] = str(MODEL_DIR)

# ── 训练超参 ──
cfg['train']['batch_size']        = TRAIN_BATCH_SIZE
cfg['train']['epochs']            = TRAIN_EPOCHS
cfg['train']['steps_per_epoch']   = TRAIN_STEPS_PER_EPOCH
cfg['train']['num_workers']       = TRAIN_NUM_WORKERS
cfg['train']['resume']            = TRAIN_RESUME

# ── 保存 ──
run_cfg = RUN_DIR / 'config.yaml'
RUN_DIR.mkdir(parents=True, exist_ok=True)
yaml.safe_dump(cfg, open(run_cfg, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)

print('── 运行配置预览 ──')
print(open(run_cfg, 'r', encoding='utf-8').read())

## 6. 训练

支持断点续训：`resume=True` 时自动读取 `last.pt`。
验证集 macro-F1 最优的 checkpoint 保存为 `best.pt`。

In [ ]:
!python -m seedvii_contrastive.scripts.train_contrastive --config {run_cfg}

## 7. EEG 编码 / 推理

分类时使用 80 条 L2 文本按三分类聚合得到 LLM 文本原型，再与 EEG embedding 做相似度。

In [ ]:
BEST_CKPT = RUN_DIR / 'best.pt'
OUT_EMB   = RUN_DIR / 'val_embeddings.npz'

if BEST_CKPT.exists():
    !python -m seedvii_contrastive.scripts.encode_eeg \
        --config {run_cfg} \
        --checkpoint {BEST_CKPT} \
        --split val \
        --out {OUT_EMB}
    print(f'[OK] Embeddings saved: {OUT_EMB}')
else:
    print(f'[WARN] Best checkpoint not found: {BEST_CKPT}')
    print('Training may need to complete first.')

## 8. 结果可视化（可选）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

if OUT_EMB.exists():
    data = np.load(OUT_EMB)
    embeddings = data['embedding']
    labels = data['label']
    
    # t-SNE 降维可视化
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    emb_2d = tsne.fit_transform(embeddings)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#e74c3c', '#95a5a6', '#2ecc71']
    names = ['Negative', 'Neutral', 'Positive']
    for c in range(3):
        mask = labels == c
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
                   c=colors[c], label=names[c], alpha=0.5, s=8)
    ax.legend()
    ax.set_title('t-SNE of EEG Embeddings (Validation Set)')
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'tsne_embeddings.png', dpi=150)
    plt.show()
else:
    print('No embedding file yet. Run encoding first.')

---

## 本地运行注意事项

- **路径**：所有路径通过顶部的用户配置区集中管理，支持绝对路径和相对路径。
- **GPU 显存**：`TRAIN_BATCH_SIZE` 按需调整。Qwen2.5-0.5B + EEGNet 在 8GB 显存约可跑到 batch=32。
- **断点续训**：`TRAIN_RESUME=True` 时自动检测 `last.pt`，中断后重新运行本 Notebook 即可继续。
- **跳过已完成步骤**：数据集下载、NPZ 预处理均检查产物是否存在，避免重复计算。
- **CPU 模式**：设置 `TRAIN_DEVICE='cpu'` 并将 `TRAIN_NUM_WORKERS=0`，但训练会显著变慢。
- **强制重处理**：删除对应目录（如 `NPZ_DIR`）后重新运行即可。